# 实验 03：正则表达式与网络结构化数据

**学号**：20244080326　　**姓名**：罗娜

本实验在独立的 conda 虚拟环境 `conda-hw`（Python 3.11）中完成，内容包括：

1. 正则表达式的基础匹配（字符类、量词、词边界、分支选择）；
2. 分组、后向引用与替换；
3. 用 BeautifulSoup 解析 HTML 字符串（不依赖网络下载）。

每个任务先用 Markdown 说明匹配规则，再在代码单元中完成计算并输出关键结果。

In [ ]:
import hashlib
import os
import platform
import sys
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080326

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV") or os.path.basename(sys.prefix)
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹]   {MACHINE_FP}")
print(f"[RUN_ID]     {RUN_ID}")
print(f"[Python]     {sys.version.split()[0]}")


[Conda 环境] conda-hw
[学号]       20244080326
[机器指纹]   e8e3a7d29233976b
[RUN_ID]     20244080326-e8e3a7d29233976b
[Python]     3.11.17


## 实验 1：正则表达式的基础匹配

导入 `re` 后，用 `re.findall(pattern, text)` 返回文本中所有不重叠的匹配项（若模式含捕获组，则返回组内容）。

三个元字符的含义：

- `\d+`：一个或多个数字字符（等价于 `[0-9]+`）；
- `\s+`：一个或多个空白字符（空格、制表符、换行等）；
- `\bcat\b`：`\b` 是词边界，保证 `cat` 前后都不是单词字符（字母、数字、下划线），因此不会匹配到 `cats` 中的 `cat`。

In [2]:
import re

text = "cat on mat; later: cats and dogs"

digits = re.findall(r"\d+", text)
spaces = re.findall(r"\s+", text)
cats = re.findall(r"\bcat\b", text)

print("原文本：", text)
print(r"1) \d+      ->", digits, f"（共 {len(digits)} 个）")
print(r"2) \s+      ->", [repr(s) for s in spaces], f"（共 {len(spaces)} 个空白块）")
print(r"3) \bcat\b  ->", cats, f"（共 {len(cats)} 个）")

原文本： cat on mat; later: cats and dogs
1) \d+      -> [] （共 0 个）
2) \s+      -> ["' '", "' '", "' '", "' '", "' '", "' '"] （共 6 个空白块）
3) \bcat\b  -> ['cat'] （共 1 个）


**结果说明**

- 文本中没有数字，故 `\d+` 返回空列表；
- 文本共有 7 个单词，相邻单词之间各有一个空格，共 6 个空白块；
- `cats` 中的 `cat` 后面紧跟字母 `s`，不满足右词边界，故 `\bcat\b` 只匹配到独立的 `cat`。

In [3]:
text2 = "cat dog bird CAT"

print("原文本：", text2)
print("cat|dog（区分大小写）：", re.findall(r"cat|dog", text2))
print("cat|dog（忽略大小写）：", re.findall(r"(?i)cat|dog", text2))

s = "aaaa"
print('findall(r"a|aa|aaa", "aaaa") ->', re.findall(r"a|aa|aaa", s))
print('findall(r"aaa|aa|a", "aaaa") ->', re.findall(r"aaa|aa|a", s))

原文本： cat dog bird CAT
cat|dog（区分大小写）： ['cat', 'dog']
cat|dog（忽略大小写）： ['cat', 'dog', 'CAT']
findall(r"a|aa|aaa", "aaaa") -> ['a', 'a', 'a', 'a']
findall(r"aaa|aa|a", "aaaa") -> ['aaa', 'a']


**结果说明**

- `cat|dog` 是分支选择，默认区分大小写，因此大写 `CAT` 不被匹配；加上内联标志 `(?i)` 后忽略大小写。
- 分支选择采用**最左优先**（leftmost-first），而不是最长匹配：
  - 模式 `a|aa|aaa` 在 `"aaaa"` 中每次只吃掉 1 个 `a`，得到 4 个 `a`；
  - 模式 `aaa|aa|a` 按书写顺序先尝试 `aaa`，得到 `'aaa'` 与剩余 `'a'` 两个匹配。
  
  可见**分支的书写顺序会影响匹配结果**。

In [4]:
text3 = "ct cat caat caaat"
patterns = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]

print("原文本：", text3)
for p in patterns:
    print(f"{p:<12}-> {re.findall(p, text3)}")

原文本： ct cat caat caaat
ca*t        -> ['ct', 'cat', 'caat', 'caaat']
ca+t        -> ['cat', 'caat', 'caaat']
ca?t        -> ['ct', 'cat']
ca{2}t      -> ['caat']
ca{1,2}t    -> ['cat', 'caat']


**各量词的含义**

| 模式 | 含义 | 匹配结果 |
| --- | --- | --- |
| `ca*t` | `a` 出现 **0 次或多次**（`*`） | `ct, cat, caat, caaat` |
| `ca+t` | `a` 出现 **1 次或多次**（`+`） | `cat, caat, caaat` |
| `ca?t` | `a` 出现 **0 次或 1 次**（`?`） | `ct, cat` |
| `ca{2}t` | `a` **恰好 2 次** | `caat` |
| `ca{1,2}t` | `a` **1 到 2 次** | `cat, caat` |

`caat` / `caaat` 中多余或不足的 `a` 个数不满足对应量词的要求，因此不会被部分匹配（正则从每个位置尝试，若整体失败就右移起点）。

## 实验 2：分组、后向引用与替换

**任务 1 规则**：用圆括号建立捕获组。`([\w.+-]+)` 匹配用户名（字母、数字、下划线、点、加号、减号），`@` 分隔，`([\w-]+(?:\.[\w-]+)+)` 匹配域名——内层 `(?:...)` 是非捕获组，只用于重复而不额外产生分组。

In [5]:
text_email = " 联系：alice@example.com；备选：bob@test.org"
pattern_email = r"([\w.+-]+)@([\w-]+(?:\.[\w-]+)+)"

print("原文本：", text_email)
for i, (user, domain) in enumerate(re.findall(pattern_email, text_email), 1):
    print(f"邮箱{i}：用户名 = {user:<6} 域名 = {domain}")

原文本：  联系：alice@example.com；备选：bob@test.org
邮箱1：用户名 = alice  域名 = example.com
邮箱2：用户名 = bob    域名 = test.org


**任务 2**：`\b(\w+)\b\s+\1\b` 中 `\1` 是后向引用，表示“与第 1 个捕获组捕获到的内容完全相同”，因此可以找出被空白分隔的重复单词。

In [6]:
text_dup = "cat cat, dog cat, bird bird"

print("原文本：", text_dup)
print(r'findall(r"\b(\w+)\b\s+\1\b") ->', re.findall(r"\b(\w+)\b\s+\1\b", text_dup))

原文本： cat cat, dog cat, bird bird
findall(r"\b(\w+)\b\s+\1\b") -> ['cat', 'bird']


`dog cat` 虽然相邻但单词不同，不满足后向引用；`cat cat` 与 `bird bird` 是连续重复，返回重复的单词本身（捕获组内容）。

**任务 3**：日期分隔符可能是 `-`、`/`、`.`，统一写作字符类 `[-/.]`。替换表达式 `\2-\3-\1` 依次引用“月、日、年”三个捕获组，`re.sub()` 只替换匹配到的部分，因此换行符原样保留，行数不变。

In [7]:
text_date = "2024-05-01\n2024/05/02\n2024.05.03"
pattern_date = r"(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})"

text_date_new = re.sub(pattern_date, r"\2-\3-\1", text_date)

print("转换前：")
print(text_date)
print("转换后：")
print(text_date_new)
print("行数：", len(text_date.splitlines()), "->", len(text_date_new.splitlines()))

转换前：
2024-05-01
2024/05/02
2024.05.03
转换后：
05-01-2024
05-02-2024
05-03-2024
行数： 3 -> 3


**任务 4**：电话号码的分隔符为 `-` 或 `.`，三个捕获组分别对应区号、前三位、后四位；替换表达式 `(\1) \2-\3` 借用量词外的普通字符拼出 `(123) 456-7890` 形式。

In [8]:
text_phone = "联系我：123-456-7890 或 987.654.3210"
pattern_phone = r"(\d{3})[-.](\d{3})[-.](\d{4})"

print("转换前：", text_phone)
print("转换后：", re.sub(pattern_phone, r"(\1) \2-\3", text_phone))

转换前： 联系我：123-456-7890 或 987.654.3210
转换后： 联系我：(123) 456-7890 或 (987) 654-3210


## 实验 3：使用 BeautifulSoup 解析 HTML

用 `BeautifulSoup(html_doc, "html.parser")` 把字符串构建成解析树，之后可以按标签、属性查询节点，用 `get_text()` 取得可见文本。

In [9]:
from bs4 import BeautifulSoup

html_doc = """<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>"""

soup = BeautifulSoup(html_doc, "html.parser")

print("页面标题文本：", soup.title.get_text())
print("h1 标签名：", soup.h1.name)
print("h1 文本：", soup.h1.get_text())

页面标题文本： 实验课程
h1 标签名： h1
h1 文本： 数据分析


In [10]:
paras = soup.find_all("p", class_="course")
print(f"class='course' 的段落共 {len(paras)} 个：")
for i, p in enumerate(paras, 1):
    print(f"  段落{i}：{p.get_text()}")

links = soup.find_all("a")
print(f"\n链接共 {len(links)} 个：")
for a in links:
    print(f"  文字：{a.get_text()}  href：{a['href']}")

class='course' 的段落共 2 个：
  段落1：第六讲：正则表达式
  段落2：第七讲：结构化数据

链接共 2 个：
  文字：讲义六  href：https://example.org/lecture06
  文字：讲义七  href：https://example.org/lecture07


`find_all("p", class_="course")` 同时按标签名和 `class` 属性筛选；`find_all("a")` 取出全部链接，链接地址存放在属性字典 `a['href']` 中（等价于 `a.get("href")`）。

In [11]:
print("整个文档的可见文本：")
print(repr(soup.get_text()))

print("\n单个段落的文本：", repr(paras[0].get_text()))
print("单个链接的文本：", repr(links[0].get_text()))

整个文档的可见文本：
'实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n'

单个段落的文本： '第六讲：正则表达式'
单个链接的文本： '讲义六'


**比较结果**

- 对整个文档调用 `get_text()` 会拼接所有文本节点，并保留 HTML 源码中的换行符与空白（如 `<title>` 与 `<h1>` 之间的换行），因此结果是一行一个文本片段；
- 对单个节点（段落、链接）调用 `get_text()`，只返回该节点内部的纯文本，不含任何标签；
- 三种用法都去掉了标签，但**文本范围不同**：文档级是全文，节点级只取该元素内部的内容。

## 实验小结

1. `\d+`、`\s+`、`\bcat\b` 分别按字符类与词边界匹配，`\b` 能避免匹配到更长单词内部的子串；
2. 分支选择是最左优先，书写顺序会改变 `findall` 的结果；
3. `*`、`+`、`?`、`{m}`、`{m,n}` 控制前一个字符的重复次数；
4. 捕获组配合 `re.findall` 可结构化提取字段，后向引用 `\1` 可检测重复内容，`re.sub` 借助 `\1`、`\2` 完成格式重排；
5. BeautifulSoup 的 `find_all` 支持标签 + 属性定位，`get_text()` 用于提取可见文本。